# 🦷 DENTEX v2: Xuất bộ dữ liệu Augmented (không train) để tạo Kaggle Dataset

Notebook này chạy **đúng phần dữ liệu của notebook 05** (cùng mã, cùng seed nên ra dữ liệu y hệt): nạp DENTEX gốc → chia train/val → Tooth Bank → ghép răng thật (polygon + FDI) → QC. Sau đó **lưu toàn bộ vào Output** và dừng, không train.

### Cách chạy
1. **Add Input**: dataset DENTEX có `train_quadrant_enumeration_disease.json` và ảnh.
2. **Settings**: *Internet* = **On** (để clone code). Không cần GPU: Accelerator = **None** là đủ (augment chạy trên CPU, khoảng 40 phút).
3. **Save Version → Save & Run All (Commit)**.
4. Khi xong: mở version → tab **Output** → **New Dataset** (hoặc *Create Dataset*) để biến Output thành Kaggle Dataset.

### Output
Output chỉ có `<RUN_TAG>.zip` (toàn bộ dataset) và `<RUN_TAG>_info/` (README, dataset_info, ảnh QC để xem nhanh). Kaggle có thể tự giải nén zip khi tạo Dataset; nếu không, notebook dùng dataset chỉ cần giải nén vào `/tmp`:
```python
import zipfile; zipfile.ZipFile('/kaggle/input/<dataset>/dentex_v2_augmented.zip').extractall('/tmp')
```

### Nội dung bên trong zip (`<RUN_TAG>/`)
```
images/train/          ảnh gốc của tập train
images/val/            ảnh gốc của tập val (không bao giờ bị augment)
images/synthetic/      ảnh đã ghép răng syn_<image_id>_<variant>.png
annotations/instances_train.json       nhãn train gốc (COCO, có fdi + polygon)
annotations/instances_train_aug.json   nhãn train gốc + ảnh ghép (dùng để train augmented)
annotations/instances_val.json         nhãn val
tooth_bank/            patch, mask, roi mask từng răng bệnh + bank.json
augment_state/         recipes.jsonl (tái lập từng ảnh ghép), manifest, qc_summary
qc_before_after.png, labels_check.png, bank_report.json, dataset_info.json, README.md
```
Mỗi `file_name` trong file nhãn là tên file trần; ảnh nằm ở một trong ba thư mục con của `images/` (loader của repo tìm đệ quy theo tên).


## ⚙️ Cell 0: Cấu hình


In [ ]:
DATA_ROOT = '/kaggle/input'          # nơi Kaggle gắn các dataset (tự tìm file nhãn DENTEX trong đây)
OUTPUT_ROOT = '/kaggle/working'      # được lưu thành Output của version
RUN_TAG = 'dentex_v2_augmented'      # tên thư mục dataset trong OUTPUT_ROOT
WORK_DIR = '/tmp/dentex_work'        # mã nguồn và file tạm

REPO_URL = 'https://github.com/tmanhococ/dentex-benchmark.git'
BRANCH = 'main'

NUM_AUGMENT_PER_IMAGE = 2            # số ảnh ghép sinh ra cho mỗi ảnh train (giống notebook 05)
VAL_SPLIT = 0.2                      # giống notebook 05 để ra đúng cùng tập train/val
SEED = 42
QC_PAIRS = 8                         # số cặp trước/sau hiển thị ở bước QC
ZIP_OUTPUT = True                    # True: nén cả dataset thành 1 file zip rồi xoá thư mục (Kaggle không lưu trọn hàng nghìn file)


## 💻 Cell 1: Môi trường & thư mục output


In [ ]:
import os, sys, json, time, shutil, socket
from pathlib import Path

NB_START = time.time()
IN_KAGGLE = Path('/kaggle/working').is_dir()
OUT = Path(OUTPUT_ROOT) / RUN_TAG
if OUT.exists():
    shutil.rmtree(OUT)  # luôn xuất lại từ đầu để dataset không lẫn file cũ
OUT.mkdir(parents=True)
print('Dataset sẽ lưu tại:', OUT)


## 📥 Cell 2: Lấy mã nguồn mới nhất (cần Internet = On)


In [ ]:
import subprocess

def internet_ok(host='github.com', port=443, timeout=5):
    try:
        socket.create_connection((host, port), timeout=timeout).close()
        return True
    except OSError:
        return False

if IN_KAGGLE:
    if not internet_ok():
        raise RuntimeError('Không có Internet. Settings → Internet → On (cần xác minh số điện thoại tài khoản Kaggle), rồi chạy lại.')
    REPO_DIR = Path(WORK_DIR) / 'dentex-benchmark'
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    if not (REPO_DIR / 'dentex_bench').is_dir():
        subprocess.run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '-q', 'origin', BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '-q', '--hard', f'origin/{BRANCH}'], check=True)
    os.chdir(str(REPO_DIR))
else:
    REPO_DIR = Path('.').resolve()
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
commit = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip()
print('Thư mục làm việc:', Path.cwd(), '| commit:', commit or '(không phải git repo)')
(OUT / 'code_commit.txt').write_text(commit + '\n', encoding='utf-8')


## 📦 Cell 3: Thư viện


In [ ]:
if IN_KAGGLE:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pycocotools', 'pyyaml', 'tabulate', 'scipy'], check=True)
import cv2
cv2.setNumThreads(0)
from tqdm import tqdm
print('OK | opencv', cv2.__version__)


## 🔍 Cell 4: Tìm & nạp DENTEX gốc trong `/kaggle/input` (giữ polygon + FDI)

Tìm file json có `categories_3` (nhãn bệnh), ưu tiên đường dẫn chứa `disease`. Chỉ copy đúng các ảnh mà file json liệt kê, và **kiểm tra kích thước từng ảnh khớp với nhãn** (DENTEX có nhiều tập trùng tên `train_0.png`...). Output của version trước (thư mục `RUN_TAG`) được bỏ qua khi tìm.


In [ ]:
from collections import Counter
from PIL import Image
from dentex_bench.data.parser import convert_dentex_annotations_to_coco

data_root = Path(DATA_ROOT)
if not data_root.exists():
    raise FileNotFoundError(f'Không thấy {data_root}. Hãy Add Input dataset DENTEX.')

def skip(p):
    return RUN_TAG in p.parts

# 1. File nhãn bệnh DENTEX gốc
candidates, seen_json = [], []
for j in data_root.rglob('*.json'):
    if skip(j) or 'instances' in j.name or j.stat().st_size < 1_000:
        continue
    seen_json.append(str(j))
    try:
        with open(j, 'r', encoding='utf-8') as f:
            d = json.load(f)
    except Exception:
        continue
    if isinstance(d, dict) and 'categories_3' in d and d.get('annotations'):
        candidates.append(j)
if not candidates:
    raise FileNotFoundError('Không tìm thấy file json DENTEX có categories_3 (vd train_quadrant_enumeration_disease.json) '
                            f'trong {data_root}. Các json đã thấy: {seen_json[:10]}')
candidates.sort(key=lambda p: (0 if 'disease' in str(p).lower() else 1, -p.stat().st_size))
RAW_JSON = candidates[0]
print('File nhãn:', RAW_JSON)
with open(RAW_JSON, 'r', encoding='utf-8') as f:
    raw = json.load(f)
raw_images = {im['file_name']: im for im in raw['images']}

# 2. Thư mục ảnh: thư mục chứa nhiều ảnh trùng tên nhất, ưu tiên có 'disease'
by_dir = Counter()
for p in data_root.rglob('*'):
    if p.suffix.lower() in ('.png', '.jpg', '.jpeg') and p.name in raw_images and not skip(p):
        by_dir[p.parent] += 1
if not by_dir:
    raise FileNotFoundError('Không tìm thấy ảnh nào trùng tên với file nhãn.')
IMG_SRC = sorted(by_dir, key=lambda d: (-by_dir[d], 0 if 'disease' in str(d).lower() else 1))[0]
print(f'Thư mục ảnh: {IMG_SRC} ({by_dir[IMG_SRC]}/{len(raw_images)} ảnh)')

# 3. Copy đúng các ảnh đó vào data/raw (làm sạch trước để không lẫn ảnh tập khác)
local_raw, local_proc = Path('data/raw'), Path('data/processed')
if local_raw.exists():
    shutil.rmtree(local_raw)
local_raw.mkdir(parents=True)
local_proc.mkdir(parents=True, exist_ok=True)
missing, mismatched = [], []
for name, im in tqdm(raw_images.items(), desc='copy ảnh', file=sys.stdout, mininterval=10, ncols=100):
    src = IMG_SRC / name
    if not src.is_file():
        missing.append(name)
        continue
    with Image.open(src) as pim:
        if (pim.width, pim.height) != (im['width'], im['height']):
            mismatched.append((name, pim.size, (im['width'], im['height'])))
            continue
    shutil.copy2(src, local_raw / name)
if mismatched:
    raise ValueError(f'{len(mismatched)} ảnh có kích thước khác nhãn (sai tập?), ví dụ: {mismatched[:3]}')
if missing:
    print(f'⚠️ Thiếu {len(missing)} ảnh, ví dụ {missing[:5]} — các ảnh này sẽ bị loại.')
    raw['images'] = [im for im in raw['images'] if im['file_name'] not in set(missing)]
    keep = {im['id'] for im in raw['images']}
    raw['annotations'] = [a for a in raw['annotations'] if a['image_id'] in keep]
    RAW_JSON_USED = local_proc / 'raw_filtered.json'
    RAW_JSON_USED.write_text(json.dumps(raw), encoding='utf-8')
else:
    RAW_JSON_USED = RAW_JSON

TRAIN_ANN, VAL_ANN = local_proc / 'instances_train.json', local_proc / 'instances_val.json'
split = convert_dentex_annotations_to_coco(RAW_JSON_USED, TRAIN_ANN, VAL_ANN, split_ratio=1.0 - VAL_SPLIT, seed=SEED)
print('Chia tập:', split)

CLASS_NAMES = {1: 'Caries', 2: 'Deep Caries', 3: 'Periapical Lesion', 4: 'Impacted Tooth'}
for name, p in [('train', TRAIN_ANN), ('val', VAL_ANN)]:
    c = json.loads(p.read_text(encoding='utf-8'))
    anns = c['annotations']
    per_cls = Counter(CLASS_NAMES[a['category_id']] for a in anns)
    fdi_cov = sum(1 for a in anns if a.get('fdi')) / max(1, len(anns))
    seg_cov = sum(1 for a in anns if a.get('segmentation')) / max(1, len(anns))
    print(f'{name}: {len(c["images"])} ảnh, {len(anns)} nhãn {dict(per_cls)} | có FDI: {fdi_cov:.1%} | có polygon: {seg_cov:.1%}')
    if fdi_cov < 0.9 or seg_cov < 0.9:
        raise ValueError(f'Tập {name} thiếu FDI/polygon ({fdi_cov:.1%}/{seg_cov:.1%}): file nhãn không phải DENTEX disease gốc.')
shutil.copy2(TRAIN_ANN, OUT / 'instances_train.json')
shutil.copy2(VAL_ANN, OUT / 'instances_val.json')


### Kiểm tra nhãn bằng mắt: polygon + số FDI vẽ trên 2 ảnh train


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

train_coco = json.loads(TRAIN_ANN.read_text(encoding='utf-8'))
anns_by_img = {}
for a in train_coco['annotations']:
    anns_by_img.setdefault(a['image_id'], []).append(a)
COLORS = {1: (255, 200, 0), 2: (255, 80, 0), 3: (255, 0, 200), 4: (0, 200, 255)}
fig, axes = plt.subplots(1, 2, figsize=(22, 6))
for ax, im in zip(axes, train_coco['images'][:2]):
    img = cv2.cvtColor(cv2.imread(str(local_raw / im['file_name'])), cv2.COLOR_BGR2RGB)
    for a in anns_by_img.get(im['id'], []):
        for poly in a.get('segmentation', []):
            pts = np.array(poly, dtype=np.int32).reshape(-1, 2)
            cv2.polylines(img, [pts], True, COLORS[a['category_id']], 3)
        x, y = int(a['bbox'][0]), int(a['bbox'][1])
        cv2.putText(img, f"{a.get('fdi')} {CLASS_NAMES[a['category_id']]}", (x, max(30, y - 8)), cv2.FONT_HERSHEY_SIMPLEX, 1.2, COLORS[a['category_id']], 3)
    ax.imshow(img); ax.set_title(im['file_name']); ax.axis('off')
plt.tight_layout(); plt.savefig(OUT / 'labels_check.png', dpi=60); plt.show()


## 🦷 Cell 5: Tooth Bank (mỗi răng bệnh cắt theo đúng polygon)


In [ ]:
from dentex_bench.augment.bank_builder import build_tooth_bank
from dentex_bench.augment.io_utils import read_image_lossless

BANK_DIR = Path('outputs/augment/bank')
if BANK_DIR.exists():
    shutil.rmtree(BANK_DIR)
t0 = time.time()
bank, bank_report = build_tooth_bank(TRAIN_ANN, local_raw, BANK_DIR, val_ann=VAL_ANN)
print(json.dumps(bank_report, indent=2, ensure_ascii=False))
print(f'⏱️ Tooth bank: {time.time() - t0:.0f}s')
(OUT / 'bank_report.json').write_text(json.dumps(bank_report, indent=2), encoding='utf-8')

show = bank.entries[:: max(1, len(bank.entries) // 8)][:8]
fig, axes = plt.subplots(1, len(show), figsize=(3 * len(show), 4))
for ax, e in zip(np.atleast_1d(axes), show):
    patch = read_image_lossless(bank.resolve_path(e.patch_path))
    patch = cv2.cvtColor(patch, cv2.COLOR_GRAY2RGB) if patch.ndim == 2 else patch.copy()
    roi = read_image_lossless(bank.resolve_path(e.roi_mask_path))
    contours, _ = cv2.findContours((roi > 0).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(patch, contours, -1, (255, 0, 0), 2)
    ax.imshow(patch); ax.set_title(f'FDI {e.fdi}\n{",".join(e.labels)}', fontsize=9); ax.axis('off')
plt.tight_layout(); plt.show()


## 🧬 Cell 6: Sinh dữ liệu ghép răng (có thanh tiến trình)

Tiến độ được ghi sau mỗi ảnh; nếu kernel restart, chạy lại cell này sẽ tiếp tục từ ảnh dở dang. Ảnh augment nằm trong `WORK_DIR` (không lưu vào Output vì rất nặng và sinh lại y hệt được); nhãn, recipes và QC được lưu vào `OUT/augmented_data`.


In [ ]:
from dentex_bench.augment.config import load_config
from dentex_bench.augment.offline import run_offline

aug_cfg = load_config('configs/augment/base.yaml', cli_overrides=[
    f'paths.train_ann={TRAIN_ANN}', f'paths.val_ann={VAL_ANN}', f'paths.img_dir={local_raw}',
    'paths.out_root=outputs/augment', f'global_seed={SEED}',
    f'offline.variants_per_image={NUM_AUGMENT_PER_IMAGE}', 'offline.num_workers=1',
    'output.formats=[coco]', 'output.include_originals=true',
])
aug_cfg.output.overwrite = False
t0 = time.time()
RUN_DIR = run_offline(aug_cfg, bank=bank)
print(f'⏱️ Augment: {(time.time() - t0) / 60:.1f} phút')
AUG_TRAIN_ANN = RUN_DIR / 'dataset' / 'coco' / 'instances_train_aug.json'
aug_coco = json.loads(AUG_TRAIN_ANN.read_text(encoding='utf-8'))
qc_path = RUN_DIR / 'state' / 'qc_summary.json'
qc_summary = json.loads(qc_path.read_text(encoding='utf-8')) if qc_path.exists() else {}
n_syn = sum(1 for im in aug_coco['images'] if im['file_name'].startswith('syn_'))
print(f'Tập augmented: {len(aug_coco["images"])} ảnh ({n_syn} ảnh ghép), {len(aug_coco["annotations"])} nhãn')
print('QC:', qc_summary)

aug_out = OUT / 'augmented_data'
aug_out.mkdir(parents=True, exist_ok=True)
shutil.copy2(AUG_TRAIN_ANN, aug_out / 'instances_train_aug.json')
shutil.copytree(RUN_DIR / 'state', aug_out / 'state', dirs_exist_ok=True)
print('Đã lưu nhãn/recipes vào', aug_out)


## 🔬 Cell 7: QC trực quan trước/sau

Mỗi hàng: răng nguồn (từ bank) | vùng răng đích **trước** (viền xanh = răng bị thay) | **sau** khi ghép (viền đỏ = nhãn mới) | toàn ảnh sau. Ảnh được lưu thành `OUT/qc_before_after.png`.


In [ ]:
recipes = [json.loads(l) for l in (RUN_DIR / 'state' / 'recipes.jsonl').read_text(encoding='utf-8').splitlines() if l.strip()]
syn_by_file = {im['file_name']: im['id'] for im in aug_coco['images']}
aug_anns_by_img = {}
for a in aug_coco['annotations']:
    aug_anns_by_img.setdefault(a['image_id'], []).append(a)

def to_rgb(x):
    return cv2.cvtColor(x, cv2.COLOR_GRAY2RGB) if x.ndim == 2 else x.copy()

pairs = recipes[:: max(1, len(recipes) // QC_PAIRS)][:QC_PAIRS]
fig, axes = plt.subplots(len(pairs), 4, figsize=(20, 4.5 * len(pairs)))
axes = np.atleast_2d(axes)
for row, rec in zip(axes, pairs):
    op = rec['ops'][0]
    syn_name = f"syn_{rec['target_image_id']}_{rec['rng_key'][-1]}.png"
    before = to_rgb(read_image_lossless(local_raw / rec['target_file']))
    after = to_rgb(read_image_lossless(RUN_DIR / 'synthetic' / syn_name))
    x, y, w, h = [int(v) for v in op['target_tooth_ref']['bbox']]
    for poly in op['target_tooth_ref'].get('segmentation') or []:
        cv2.polylines(before, [np.array(poly, dtype=np.int32).reshape(-1, 2)], True, (0, 255, 0), 2)
    for a in aug_anns_by_img.get(syn_by_file.get(syn_name), []):
        if a.get('synthetic'):
            for poly in a.get('segmentation') or []:
                cv2.polylines(after, [np.array(poly, dtype=np.int32).reshape(-1, 2)], True, (255, 0, 0), 2)
    pad = int(0.6 * max(w, h))
    y1, y2, x1, x2 = max(0, y - pad), y + h + pad, max(0, x - pad), x + w + pad
    src = bank.get(op['source_entry_id'])
    row[0].imshow(to_rgb(read_image_lossless(bank.resolve_path(src.patch_path))))
    row[0].set_title(f"Nguồn {src.entry_id} ({','.join(src.labels)})" + (' [lật]' if op['mirror'] else ''), fontsize=9)
    row[1].imshow(before[y1:y2, x1:x2]); row[1].set_title(f"Trước: FDI {op['target_fdi']} ({rec['target_file']})", fontsize=9)
    row[2].imshow(after[y1:y2, x1:x2]); row[2].set_title('Sau khi ghép', fontsize=9)
    small = after.copy(); cv2.rectangle(small, (x, y), (x + w, y + h), (255, 0, 0), 6)
    row[3].imshow(small); row[3].set_title(syn_name, fontsize=9)
    for ax in row:
        ax.axis('off')
plt.tight_layout()
plt.savefig(OUT / 'qc_before_after.png', dpi=80)
plt.show()


## 💾 Cell 8: Ghi bộ dữ liệu vào Output

Copy ảnh (có thanh tiến trình), file nhãn, tooth bank và recipes vào `OUT`, rồi **kiểm tra mọi ảnh mà file nhãn nhắc tới đều có mặt**.


In [ ]:
from collections import Counter

IMG_OUT = OUT / 'images'
ANN_OUT = OUT / 'annotations'
for d in [IMG_OUT / 'train', IMG_OUT / 'val', IMG_OUT / 'synthetic', ANN_OUT]:
    d.mkdir(parents=True, exist_ok=True)

def copy_many(names, src_dir, dst_dir, desc):
    for name in tqdm(names, desc=desc, file=sys.stdout, mininterval=10, ncols=100):
        src = Path(src_dir) / name
        if not src.is_file():
            raise FileNotFoundError(f'{desc}: thiếu {src}')
        shutil.copy2(src, Path(dst_dir) / name)

train_orig = json.loads(TRAIN_ANN.read_text(encoding='utf-8'))
val_coco = json.loads(VAL_ANN.read_text(encoding='utf-8'))
syn_names = [im['file_name'] for im in aug_coco['images'] if im['file_name'].startswith('syn_')]
copy_many([im['file_name'] for im in train_orig['images']], local_raw, IMG_OUT / 'train', 'ảnh train')
copy_many([im['file_name'] for im in val_coco['images']], local_raw, IMG_OUT / 'val', 'ảnh val')
copy_many(syn_names, RUN_DIR / 'synthetic', IMG_OUT / 'synthetic', 'ảnh ghép')

shutil.copy2(TRAIN_ANN, ANN_OUT / 'instances_train.json')
shutil.copy2(AUG_TRAIN_ANN, ANN_OUT / 'instances_train_aug.json')
shutil.copy2(VAL_ANN, ANN_OUT / 'instances_val.json')
shutil.copytree(BANK_DIR, OUT / 'tooth_bank')
shutil.copytree(RUN_DIR / 'state', OUT / 'augment_state')
# Bản sao tạm do các cell dùng chung tạo ra; nội dung đã nằm trong annotations/ và augment_state/
shutil.rmtree(OUT / 'augmented_data', ignore_errors=True)
for f in ['instances_train.json', 'instances_val.json']:
    (OUT / f).unlink(missing_ok=True)

# Kiểm tra: mọi ảnh trong nhãn đều có file, và train/val không trùng ảnh
on_disk = {p.name for p in IMG_OUT.rglob('*') if p.is_file()}
for ann_name in ['instances_train.json', 'instances_train_aug.json', 'instances_val.json']:
    c = json.loads((ANN_OUT / ann_name).read_text(encoding='utf-8'))
    lost = [im['file_name'] for im in c['images'] if im['file_name'] not in on_disk]
    if lost:
        raise FileNotFoundError(f'{ann_name}: {len(lost)} ảnh không có trong images/, ví dụ {lost[:3]}')
overlap = {im['file_name'] for im in train_orig['images']} & {im['file_name'] for im in val_coco['images']}
if overlap:
    raise ValueError(f'Train và val trùng {len(overlap)} ảnh: {sorted(overlap)[:3]}')
print('✅ Mọi ảnh trong 3 file nhãn đều có mặt; train/val không trùng ảnh.')


## 🧾 Cell 9: Thông tin dataset, README, kiểm tra dung lượng


In [ ]:
CLASS_NAMES = {1: 'Caries', 2: 'Deep Caries', 3: 'Periapical Lesion', 4: 'Impacted Tooth'}

def summarize(c):
    return {'images': len(c['images']), 'annotations': len(c['annotations']),
            'per_class': dict(Counter(CLASS_NAMES[a['category_id']] for a in c['annotations']))}

def dir_size_mb(d):
    return round(sum(p.stat().st_size for p in Path(d).rglob('*') if p.is_file()) / 1024**2, 1)

info = {
    'source_label_file': str(RAW_JSON),
    'code_commit': commit,
    'seed': SEED, 'val_split': VAL_SPLIT, 'variants_per_image': NUM_AUGMENT_PER_IMAGE,
    'splits': {'train': summarize(train_orig), 'train_aug': summarize(aug_coco), 'val': summarize(val_coco)},
    'synthetic_images': len(syn_names),
    'tooth_bank_entries': len(bank.entries),
    'augment_qc': qc_summary,
    'size_mb': {d: dir_size_mb(OUT / d) for d in ['images/train', 'images/val', 'images/synthetic', 'annotations', 'tooth_bank', 'augment_state']},
    'created': time.strftime('%Y-%m-%d %H:%M:%S'),
}
(OUT / 'dataset_info.json').write_text(json.dumps(info, indent=2, ensure_ascii=False), encoding='utf-8')

readme = f'''# DENTEX v2 tooth-swap augmented dataset

Built from `{RAW_JSON.name}` (DENTEX quadrant-enumeration-disease) with dentex-benchmark commit `{commit}`,
seed {SEED}, train/val split {1 - VAL_SPLIT:.0%}/{VAL_SPLIT:.0%}, {NUM_AUGMENT_PER_IMAGE} synthetic variants per train image.

Each synthetic image replaces one diseased tooth with a diseased tooth of the same FDI number (or its mirror
counterpart) cut along its DENTEX polygon from another train image. Validation images are never augmented.

| split | images | annotations |
|---|---|---|
| train (original) | {info['splits']['train']['images']} | {info['splits']['train']['annotations']} |
| train_aug (original + synthetic) | {info['splits']['train_aug']['images']} | {info['splits']['train_aug']['annotations']} |
| val | {info['splits']['val']['images']} | {info['splits']['val']['annotations']} |

Layout: `images/{{train,val,synthetic}}/`, `annotations/instances_{{train,train_aug,val}}.json` (COCO, category ids
1 Caries, 2 Deep Caries, 3 Periapical Lesion, 4 Impacted Tooth; each annotation keeps `fdi` and `segmentation`),
`tooth_bank/` (per-tooth patches and masks), `augment_state/recipes.jsonl` (exact parameters of every synthetic image).
`file_name` is a bare name; look images up under `images/` recursively.
'''
(OUT / 'README.md').write_text(readme, encoding='utf-8')

n_files = sum(1 for p in OUT.rglob('*') if p.is_file())
print(json.dumps(info, indent=2, ensure_ascii=False))
print(f'\n📦 {OUT}: {n_files} file, {dir_size_mb(OUT) / 1024:.2f} GB')


## 🗜️ Cell 10: Nén thành 1 file zip

Kaggle không lưu trọn Output có hàng nghìn file trong nhiều thư mục con (lần chạy trước: 10136 file, 5.55 GB nhưng Output chỉ còn 50 file, 117 MB). Vì vậy dataset được nén thành **một file** `<RUN_TAG>.zip` rồi xoá thư mục gốc. Ảnh PNG đã nén sẵn nên zip dùng chế độ *store* (không nén lại): nhanh và dung lượng gần như không đổi.

Một vài file nhỏ (`README.md`, `dataset_info.json`, ảnh QC) được để ngoài zip trong `<RUN_TAG>_info/` để xem nhanh trên Kaggle.


In [ ]:
import zipfile

ZIP_PATH = Path(OUTPUT_ROOT) / f'{RUN_TAG}.zip'
INFO_DIR = Path(OUTPUT_ROOT) / f'{RUN_TAG}_info'
if not ZIP_OUTPUT:
    print('ZIP_OUTPUT = False: giữ nguyên thư mục, không nén.')
else:
    free_gb = shutil.disk_usage(OUTPUT_ROOT).free / 1024**3
    need_gb = dir_size_mb(OUT) / 1024
    print(f'Cần {need_gb:.2f} GB để nén, còn trống {free_gb:.2f} GB')
    if free_gb < need_gb * 1.05:
        raise OSError('Không đủ chỗ trống để tạo file zip')

    files = sorted(p for p in OUT.rglob('*') if p.is_file())
    ZIP_PATH.unlink(missing_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_STORED, allowZip64=True) as zf:
        for p in tqdm(files, desc='nén zip', file=sys.stdout, mininterval=10, ncols=100):
            zf.write(p, arcname=str(Path(RUN_TAG) / p.relative_to(OUT)))

    # Kiểm tra zip trước khi xoá thư mục gốc
    with zipfile.ZipFile(ZIP_PATH) as zf:
        names = [n for n in zf.namelist() if not n.endswith('/')]
    if len(names) != len(files):
        raise RuntimeError(f'Zip có {len(names)} file, thư mục có {len(files)} file')

    if INFO_DIR.exists():
        shutil.rmtree(INFO_DIR)
    INFO_DIR.mkdir(parents=True)
    for f in ['README.md', 'dataset_info.json', 'bank_report.json', 'code_commit.txt',
              'qc_before_after.png', 'labels_check.png']:
        if (OUT / f).is_file():
            shutil.copy2(OUT / f, INFO_DIR / f)
    shutil.rmtree(OUT)
    print(f'✅ {ZIP_PATH.name}: {len(names)} file, {ZIP_PATH.stat().st_size / 1024**3:.2f} GB')

left = [p for p in Path(OUTPUT_ROOT).rglob('*') if p.is_file()]
print(f'Output còn {len(left)} file:', sorted(str(p.relative_to(OUTPUT_ROOT)) for p in left)[:12])
print(f'Tổng thời gian: {(time.time() - NB_START) / 60:.1f} phút. Xong: vào tab Output → New Dataset.')
